## Binning species fractions

**Purpose**

This notebook preprocesses single-cell FLIM-FRET data from MC4R experiments. Each measurement yields, per cell, a total receptor surface concentration and the fractions of receptor existing in monomer, dimer, and higher-order oligomer states. Because receptor expression varies widely across cells (spanning several orders of magnitude), data points are grouped into logarithmically spaced concentration bins before downstream model fitting. This binning reduces noise and allows the concentration-dependence of the oligomeric equilibrium to be visualized and analyzed.

**Input**
Data.txt: Tab-delimited text with one row per cell/measurement. Expected columns: total_conc (surface receptor concentration), donor_fraction, monomer_frac, dimer_frac, oligomer_frac

**Workflow**
*Step 1 — Log-spaced concentration binning*

The range of total_conc values is determined from the data.
Small margins are added (×0.9 lower, ×1.1 upper) to avoid edge clipping.
A floor of 1e-6 prevents log(0) errors.
15 logarithmically spaced bins are created with np.logspace.
Each data point is assigned to a bin using pd.cut(..., include_lowest=True).

*Step 2 — Per-bin statistics*
For each bin, the following statistics are computed for monomer_frac, dimer_frac, and oligomer_frac:

| Statistic | Meaning| 
| --- | --- | 
| mean | Average species fraction in the bin | 
| std | Standard deviation (spread of single-cell values) | 
| count | Number of cells in the bin | 
| sem | Standard error of the mean (used for error bars in plots/fits) | 


Bin centers are added as the arithmetic mean of the bin edges (note: for a log-spaced binning, the geometric mean would be more natural, but the arithmetic mean is used here — worth keeping in mind for plotting on a log axis).
NaN standard deviations (arising from bins with only one data point) are replaced with 0.

*Step 3 — Output*
Results are saved to MC4R-B2_MultiInt_binned_summary.csv, ready for import into the downstream fitting notebooks.

**Output file**
Multi-level column CSV: species × statistic, plus bin_center column. One row per concentration bin.


In [1]:
import numpy as np
import pandas as pd

# Load your txt file (adjust delimiter if needed: "\t" for tab, "," for comma, " " for space)
df = pd.read_csv("Data.txt", delimiter="\t")

# Example DataFrame structure:
# df = pd.DataFrame({
#     "total_conc": [...],       # total protein concentration
#     "donor_fraction": [...],       # donor fraction
#     "monomer_frac": [...],
#     "dimer_frac": [...],
#     "oligomer_frac": [...]
# })

### (1) Bin by total protein concentration (log scale, 15 bins)
# Automatically determine log-spaced bins based on data range
n_log_bins = 15  # number of bins you want
min_val = df["total_conc"].min()
max_val = df["total_conc"].max()

# Add a small margin (optional, avoids edge clipping)
min_val = max(min_val * 0.9, 1e-6)  # avoid log(0)
max_val = max_val * 1.1

log_bins = np.logspace(np.log10(min_val), np.log10(max_val), n_log_bins + 1)
df["conc_bin"] = pd.cut(df["total_conc"], bins=log_bins, include_lowest=True)

conc_summary = df.groupby("conc_bin", observed=False)[["monomer_frac", "dimer_frac", "oligomer_frac"]].agg(["mean", "std", "count", "sem"])
# Add bin centers
conc_summary["bin_center"] = (log_bins[:-1] + log_bins[1:]) / 2

# Replace std NaN with 0 or "n/a"
conc_summary = conc_summary.fillna({"monomer_frac": {"std": 0}, "dimer_frac": {"std": 0},  "oligomer_frac": {"std": 0}})
conc_summary.to_csv("MC4R-B2_MultiInt_binned_summary.csv")

/tmp/ipykernel_1408769/21221477.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  conc_summary = df.groupby("conc_bin")[["monomer_frac", "dimer_frac", "oligomer_frac"]].agg(["mean", "std", "count", "sem"])
